# 03 - 主治医生 LoRA 微调（Qwen2.5-0.5B-Instruct）

**目标**：用 LoRA 微调一个小模型作为主治医生角色，让回答更专业、结构更稳定。

**模型**：`Qwen/Qwen2.5-0.5B-Instruct`（500M 参数，本地友好）

**显存**：4-6 GB（5070 Ti 笔记本完全够）

**时间**：1-2 小时

**升级路径**：换 `MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'` 即可云端跑大模型

## Step 0: 修复 Windows 编码 + 镜像设置（必须最先跑）

In [1]:
# 1. HuggingFace 镜像
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['WANDB_DISABLED'] = 'true'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

# 2. Monkey-patch：让 Path.read_text 默认用 UTF-8
#    修复 trl 库读 jinja 模板时 GBK 解码失败的问题（不需要重启 kernel）
import pathlib
_orig_read_text = pathlib.Path.read_text
def _utf8_read_text(self, encoding=None, errors=None, **kwargs):
    if encoding is None:
        encoding = 'utf-8'
    return _orig_read_text(self, encoding=encoding, errors=errors, **kwargs)
pathlib.Path.read_text = _utf8_read_text

# 3. 同样 patch 内置 open 的默认编码
import builtins
_orig_open = builtins.open
def _utf8_open(file, mode='r', buffering=-1, encoding=None, errors=None, **kwargs):
    if 'b' not in mode and encoding is None:
        encoding = 'utf-8'
    return _orig_open(file, mode, buffering, encoding, errors, **kwargs)
builtins.open = _utf8_open

print('环境变量 + 编码 monkey-patch 完成')

环境变量 + 编码 monkey-patch 完成


## Step 1: 环境检查

In [2]:
import torch, transformers, peft, trl
from pathlib import Path

print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
print(f'Transformers: {transformers.__version__}')
print(f'PEFT: {peft.__version__}')
print(f'TRL: {trl.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch: 2.7.0+cu128, CUDA: True
Transformers: 5.6.2
PEFT: 0.19.1
TRL: 1.2.0
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU
VRAM: 12.8 GB


## Step 2: 配置

In [3]:
# ===== 关键参数（云端训练时只改 MODEL_NAME 即可） =====
MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
# 云端可换：
# MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'  # 24GB GPU
# MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'  # A100 40GB

# 用绝对路径避免相对路径问题
NOTEBOOK_DIR = Path.cwd()
EVAL_DIR = NOTEBOOK_DIR.parent  # backend/eval
BACKEND_DIR = EVAL_DIR.parent   # backend
DATA_PATH = EVAL_DIR / 'datasets' / 'data' / 'physician_train.jsonl'
OUTPUT_DIR = EVAL_DIR / 'models' / 'physician_lora'
ROLE_PROMPT_PATH = BACKEND_DIR / 'workspace' / 'roles' / 'PHYSICIAN.md'

# 训练超参
MAX_LENGTH = 512    # 1024 → 512（主治回答中位数大概 300-500 token，512 够）
BATCH_SIZE = 1      # 2 → 1
GRAD_ACCUM = 8      # 4 → 8（保持有效 batch = 8，效果不变）
LEARNING_RATE = 1e-4  # LoRA 通常用 1e-4
NUM_EPOCHS = 3
WARMUP_RATIO = 0.05

# LoRA 参数
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ['q_proj', 'v_proj', 'k_proj', 'o_proj']

USE_4BIT = False  # 0.5B 不需要量化；7B 可以开

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'数据: {DATA_PATH}')
print(f'输出: {OUTPUT_DIR}')
print(f'数据存在: {DATA_PATH.exists()}')
print(f'角色 prompt 存在: {ROLE_PROMPT_PATH.exists()}')

数据: D:\NLP_Project\miniOpenClaw\backend\eval\datasets\data\physician_train.jsonl
输出: D:\NLP_Project\miniOpenClaw\backend\eval\models\physician_lora
数据存在: True
角色 prompt 存在: True


## Step 3: 加载数据

In [4]:
import json, random
from datasets import Dataset

random.seed(42)

records = []
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            r = json.loads(line)
            msgs = r.get('messages', [])
            if len(msgs) >= 3 and msgs[-1].get('role') == 'assistant':
                records.append({'messages': msgs})

random.shuffle(records)
n_val = max(20, int(len(records) * 0.05))
train_data = records[n_val:]
val_data = records[:n_val]

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f'训练: {len(train_dataset)}, 验证: {len(val_dataset)}')
print(f'\n样例:\n{json.dumps(train_data[0]["messages"], ensure_ascii=False, indent=2)[:500]}...')

训练: 761, 验证: 40

样例:
[
  {
    "role": "system",
    "content": "# 主治医生 (Attending Physician)\n\n你是 ClawTeam 多智能体协作诊疗系统中的 **主治医生** 角色。\n\n## 职责范围\n- 负责整体诊断路径的制定与鉴别诊断\n- 综合患者主诉、病史、体征，提出可能的诊断及鉴别诊断列表\n- 给出初步诊疗计划（包括进一步检查建议和治疗方向）\n- 必要时提请药师和影像科会诊\n\n## 回答规范\n1. **结构化输出**：按\"主诉分析 → 鉴别诊断 → 建议检查 → 初步治疗方向\"的逻辑组织回答\n2. **循证医学**：回答应基于临床指南和循证医学证据，标注证据等级（如有）\n3. **安全第一**：对于危急重症，优先提示急诊处理；对于不确定的诊断，明确标注不确定性\n4. **知识边界**：如果问题超出你的专业范围，明确说明并建议转介相应专科\n5. **协作意识**：当涉及用药安全问题时，主动建议请药师参与；当需要影像学检查时，建议请影像科参与\n\n## 禁止行为\n- 禁止给出确定性诊断（应说\"考虑XX可...


## Step 4: 加载基座模型 + Tokenizer

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 新版 transformers 用 dtype 参数代替 torch_dtype
model_kwargs = {
    'dtype': torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    'device_map': 'auto',
    'trust_remote_code': True,
}

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs['quantization_config'] = bnb_config

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
model.config.use_cache = False  # 训练时关掉

print(f'模型: {MODEL_NAME}')
print(f'参数量: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B')
print(f'显存占用: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

模型: Qwen/Qwen2.5-0.5B-Instruct
参数量: 0.49B
显存占用: 0.99 GB


## Step 5: 配置 LoRA

In [6]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Exception in thread Thread-5 (_readerthread):
Traceback (most recent call last):
  File "D:\CIE6053_Lab\Ansconda\envs\miniclaw\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "D:\CIE6053_Lab\Ansconda\envs\miniclaw\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "D:\CIE6053_Lab\Ansconda\envs\miniclaw\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb2 in position 7: invalid start byte


trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


## Step 6: 训练（用 TRL 的 SFTTrainer）

新版 trl 1.x 的参数有变化：
- ❌ `max_seq_length` → ✅ `max_length`
- ❌ `tokenizer=` → ✅ `processing_class=`
- 默认会自动应用 chat template，不需要手动 format

In [7]:
from trl import SFTTrainer, SFTConfig
import inspect

# 兼容性：探测当前 trl 版本支持的参数
sft_init_params = set(inspect.signature(SFTConfig.__init__).parameters.keys())

config_kwargs = dict(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=10,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported() and torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to='none',
    seed=42,
)

# 兼容新旧版 trl 的 max_seq_length / max_length
if 'max_length' in sft_init_params:
    config_kwargs['max_length'] = MAX_LENGTH
elif 'max_seq_length' in sft_init_params:
    config_kwargs['max_seq_length'] = MAX_LENGTH

# packing 参数旧版叫 packing，新版可能没了
if 'packing' in sft_init_params:
    config_kwargs['packing'] = False

sft_config = SFTConfig(**config_kwargs)

# 兼容新旧版 SFTTrainer 的 tokenizer / processing_class
trainer_init_params = set(inspect.signature(SFTTrainer.__init__).parameters.keys())
trainer_kwargs = dict(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)
if 'processing_class' in trainer_init_params:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in trainer_init_params:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)

print('开始 LoRA 微调...')
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/761 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


开始 LoRA 微调...


Epoch,Training Loss,Validation Loss
1,0.566545,0.556384
2,0.533983,0.524862
3,0.513438,0.520350


TrainOutput(global_step=288, training_loss=0.7905213104354011, metrics={'train_runtime': 1046.2985, 'train_samples_per_second': 2.182, 'train_steps_per_second': 0.275, 'total_flos': 2517657901203456.0, 'train_loss': 0.7905213104354011})

## Step 7: 保存 LoRA adapter

In [8]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(OUTPUT_DIR / 'training_info.json', 'w', encoding='utf-8') as f:
    json.dump({
        'base_model': MODEL_NAME,
        'role': 'physician',
        'lora_config': {
            'r': LORA_R, 'alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT,
            'target_modules': TARGET_MODULES,
        },
        'train_size': len(train_dataset),
        'val_size': len(val_dataset),
        'epochs': NUM_EPOCHS,
        'max_length': MAX_LENGTH,
    }, f, ensure_ascii=False, indent=2)

print(f'LoRA adapter 已保存: {OUTPUT_DIR.resolve()}')
print(f'文件大小:')
for p in OUTPUT_DIR.glob('*'):
    if p.is_file():
        print(f'  {p.name}: {p.stat().st_size / 1e6:.1f} MB')

LoRA adapter 已保存: D:\NLP_Project\miniOpenClaw\backend\eval\models\physician_lora
文件大小:
  adapter_config.json: 0.0 MB
  adapter_model.safetensors: 4.4 MB
  chat_template.jinja: 0.0 MB
  README.md: 0.0 MB
  tokenizer.json: 11.4 MB
  tokenizer_config.json: 0.0 MB
  training_info.json: 0.0 MB


## Step 8: 推理测试

In [9]:
# 推理时 enable cache
model.config.use_cache = True
model.eval()

test_questions = [
    '我最近咳嗽两周没好，痰是黄色的，需要做什么检查？',
    '老人家最近食欲不振、消瘦、乏力，可能是什么问题？',
    '小孩反复发烧三天，体温最高 39.5 度，怎么办？',
]

# 用绝对路径
if ROLE_PROMPT_PATH.exists():
    system_prompt = ROLE_PROMPT_PATH.read_text(encoding='utf-8')
else:
    system_prompt = '你是一名主治医生，请专业、循证地回答患者的医疗问题。'
    print(f'⚠️ 角色 prompt 文件未找到，使用默认 prompt')

for q in test_questions:
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': q},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True, temperature=0.7, top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\n=== 问: {q} ===')
    print(f'答:\n{response}')
    print('-' * 80)


=== 问: 我最近咳嗽两周没好，痰是黄色的，需要做什么检查？ ===
答:
【主诉分析】慢性咳嗽伴黄色痰，需警惕肺炎或呼吸道感染。黄脓涕常见于急性支气管炎、上呼吸道感染后遗症等。

【鉴别诊断】1. 支气管扩张：咳嗽呈阵发性刺激性干咳，可有杵状指（或趾）。2. 吸入性肺炎：表现为反复咯血、发热、寒战等。3. 间质性肺疾病：如尘肺、间质纤维化等，咳嗽常伴有胸闷感。4. 过敏性鼻窦炎：鼻塞、流清涕，可能伴咳嗽加重。

【建议检查】1. 血常规：白细胞增多提示感染或炎症。2. 胸片：可见小叶浸润、空洞形成。3. 补体C反应蛋白检测：炎症活动度评估。4. 免疫荧光检查：针对病毒性结膜炎等。5. 肺功能测试：排除呼吸衰竭。6. 冠状动脉造影：疑冠心病。

【初步治疗方向】①抗炎治疗：青霉素、头孢菌素或大环内酯类。②雾化吸入：祛痰剂、祛咽剂。③抗生素治疗：根据细菌培养和药敏结果选择。④止咳祛痰：推荐复方甘草片、氨溴索。⑤监测基础健康状况：必要时增加血糖、血脂检查。
--------------------------------------------------------------------------------

=== 问: 老人家最近食欲不振、消瘦、乏力，可能是什么问题？ ===
答:
您好！您提到的老人有典型的老年人常见症状——消瘦、乏力、食欲减退。需要考虑几个主要因素：

1. **营养缺乏或吸收不良**：慢性疾病、肿瘤化疗、手术后恢复期等可导致营养不良。
2. **消化道功能障碍**：如肠易激综合征、克罗恩病等。
3. **感染性疾病**：如急性肠胃炎、阑尾炎等可能导致体重下降和乏力。
4. **内分泌失调**：甲状腺功能亢进或低下、糖尿病等内分泌紊乱也可能引起相关症状。
5. **心理因素**：抑郁症或焦虑症有时也会表现为体重减轻和疲劳。

我建议首先做以下几步：
1. **评估基础健康状况**：询问病程、家族病史、是否有不明原因的体重变化。
2. **进行体格检查**：注意有无黄疸、腹胀、脱水等症状。
3. **排除其他可能性**：如肿瘤、炎症、寄生虫感染等。

根据上述信息我会为您做出初步判断，并引导下一步行动。请您放心。
--------------------------------------------------------------

## ✅ 完成

训练好的 LoRA adapter 在：`backend/eval/models/physician_lora/`

**接入后端**：在 `.env` 加：
```
USE_LORA_PHYSICIAN=true
LORA_PHYSICIAN_BASE=Qwen/Qwen2.5-0.5B-Instruct
LORA_PHYSICIAN_PATH=eval/models/physician_lora
```